# 01 - Data preprocessing

Load the survey export from the project data/ folder, rename the survey items to readable snake_case names, and convert the multiple-choice responses to numeric values.

Before exporting the CSV, remove the consent, name, email, college, and program columns in the sheet. The notebook expects only the 18 predictor items and 7 AI adoption frequency items. It does not remove rows or calculate composite scores.


In [ ]:
from pathlib import Path

import pandas as pd

# Works whether Jupyter starts in the project root or the notebooks/ folder.
cwd = Path.cwd()
project_candidates = [cwd, cwd.parent]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "requirements.txt").is_file()),
    cwd,
)
DATA_DIR = PROJECT_ROOT / "data"

# If there is only one CSV in data/, it is selected automatically.
# If there are several, set CSV_FILENAME to the exact filename to process.
CSV_FILENAME = "survey_responses.csv"

if CSV_FILENAME is not None:
    CSV_PATH = DATA_DIR / CSV_FILENAME
    if not CSV_PATH.is_file():
        raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
else:
    csv_files = sorted(DATA_DIR.glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(
            f"No CSV files found in {DATA_DIR}. Copy the survey export there first."
        )
    if len(csv_files) > 1:
        raise ValueError(
            f"Found {len(csv_files)} CSV files in {DATA_DIR}. Set CSV_FILENAME above."
        )
    CSV_PATH = csv_files[0]

raw_df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
print(
    f"Loaded {len(raw_df):,} response(s) and {len(raw_df.columns)} column(s) from {CSV_PATH.name}"
)

FileNotFoundError: No CSV files found in c:\Users\Marco Bautista\ai-adoption-analysis\data. Copy the survey export there first.

## Rename survey columns

These text fragments identify each question in the Google Forms headers. Header matching uses literal text; no regular expressions are needed.


In [ ]:
COLUMN_NAME_RULES = [
    ("main functions of generative ai", "ai_familiarity_main_functions"),
    ("which generative ai tools are available", "ai_familiarity_tool_availability"),
    (
        "kinds of academic tasks generative ai can and cannot support",
        "ai_familiarity_task_fit",
    ),
    (
        "help me understand academic concepts",
        "perceived_usefulness_concept_understanding",
    ),
    (
        "complete academic tasks more efficiently",
        "perceived_usefulness_task_efficiency",
    ),
    (
        "improve the quality of some of my academic work",
        "perceived_usefulness_work_quality",
    ),
    ("learning to use generative ai tools is easy", "perceived_ease_learning"),
    (
        "using generative ai tools for academic tasks is easy",
        "perceived_ease_academic_tasks",
    ),
    ("easy for me to get a useful response", "perceived_ease_useful_responses"),
    (
        "dependable enough to assist with academic work",
        "trust_information_dependability",
    ),
    ("trust ai-generated explanations", "trust_explanations"),
    (
        "concerned that generative ai tools may provide inaccurate or misleading information",
        "trust_accuracy_concern",
    ),
    (
        "reliable access to a device and internet connection",
        "accessibility_reliable_access",
    ),
    ("can access generative ai tools that are useful", "accessibility_tool_access"),
    ("cost or access restrictions make it difficult", "accessibility_cost_barrier"),
    ("write clear instructions or prompts", "digital_literacy_prompting"),
    ("verify important ai-generated information", "digital_literacy_verification"),
    ("recognize limitations or possible bias", "digital_literacy_limitations_bias"),
    (
        "understanding or getting an explanation of an academic concept",
        "adoption_explain_concepts",
    ),
    ("summarizing or simplifying course readings", "adoption_summarize_materials"),
    ("brainstorming academic ideas", "adoption_brainstorm_ideas"),
    (
        "drafting, revising, or proofreading academic writing",
        "adoption_writing_support",
    ),
    ("creating study materials", "adoption_create_study_materials"),
    ("working through academic problems or examples", "adoption_solve_problems"),
    ("planning or organizing research tasks and information", "adoption_plan_research"),
]

rename_map = {}
unmapped_headers = []
for original_name in raw_df.columns:
    header = str(original_name).strip().casefold()
    matches = [
        new_name
        for phrase, new_name in COLUMN_NAME_RULES
        if phrase.casefold() in header
    ]

    if len(matches) != 1:
        unmapped_headers.append(str(original_name))
    else:
        rename_map[original_name] = matches[0]

if unmapped_headers:
    raise ValueError(
        "Expected only the survey item columns. These headers were not mapped; "
        "check that the sheet export excludes metadata and update COLUMN_NAME_RULES if needed: "
        + repr(unmapped_headers)
    )

renamed_df = raw_df.rename(columns=rename_map)
if renamed_df.columns.duplicated().any():
    duplicates = renamed_df.columns[renamed_df.columns.duplicated()].tolist()
    raise ValueError(f"Two or more headers map to the same name: {duplicates}")

PREDICTOR_ITEM_COLUMNS = [
    "ai_familiarity_main_functions",
    "ai_familiarity_tool_availability",
    "ai_familiarity_task_fit",
    "perceived_usefulness_concept_understanding",
    "perceived_usefulness_task_efficiency",
    "perceived_usefulness_work_quality",
    "perceived_ease_learning",
    "perceived_ease_academic_tasks",
    "perceived_ease_useful_responses",
    "trust_information_dependability",
    "trust_explanations",
    "trust_accuracy_concern",
    "accessibility_reliable_access",
    "accessibility_tool_access",
    "accessibility_cost_barrier",
    "digital_literacy_prompting",
    "digital_literacy_verification",
    "digital_literacy_limitations_bias",
]
ADOPTION_ITEM_COLUMNS = [
    "adoption_explain_concepts",
    "adoption_summarize_materials",
    "adoption_brainstorm_ideas",
    "adoption_writing_support",
    "adoption_create_study_materials",
    "adoption_solve_problems",
    "adoption_plan_research",
]
ITEM_COLUMNS = PREDICTOR_ITEM_COLUMNS + ADOPTION_ITEM_COLUMNS

missing_columns = sorted(set(ITEM_COLUMNS) - set(renamed_df.columns))
if missing_columns:
    raise ValueError(f"Survey item columns are missing: {missing_columns}")

print(f"Mapped {len(ITEM_COLUMNS)} survey item columns.")

## Convert multiple-choice responses to numbers

Each multiple-choice option starts with its numeric score (for example, 5 - Strongly Agree or 0 - Never). The code keeps the text before the hyphen separator as a number. Blank or unrecognized choices remain missing values.


In [ ]:
def choice_to_number(value):
    """Return the numeric prefix from a multiple-choice answer."""
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    score_text = text.partition(" - ")[0].strip()
    return pd.to_numeric(score_text, errors="coerce")


cleaned_df = renamed_df[ITEM_COLUMNS].copy()
for column in ITEM_COLUMNS:
    cleaned_df[column] = cleaned_df[column].map(choice_to_number).astype("Int64")

unconverted_counts = cleaned_df.isna().sum()
print(f"Cleaned data shape: {cleaned_df.shape}")
print(f"Blank or unrecognized answers: {int(unconverted_counts.sum())}")
cleaned_df.head()

## Export Cleaned Dataset

Save the cleaned numerical responses as `cleaned_data.csv` in the project's data folder.

This file will serve as the input for `02_scoring.ipynb`.


In [ ]:
EXPORT_PATH = DATA_DIR / "cleaned_data.csv"

cleaned_df.to_csv(EXPORT_PATH, index=False)

print(f"Cleaned dataset exported to: {EXPORT_PATH}")
print(f"Dataset shape: {cleaned_df.shape}")